# What people keep asking for

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fuellabs-ai/hobbylm-embed-cookbook/blob/main/what-people-ask-for/walkthrough.ipynb)

Your feedback board has hundreds of requests and many of them are the same ask
in different words. This walkthrough uses HobbyLM-Embed 140M, embeddings only,
no LLM, to group an export, count what repeats, and write a report you can
read in one sitting. Everything runs on your own laptop; nothing leaves your
machine.

The sample data in `examples/` is fictional and clearly marked.

## Setup

The constants below name the repo, branch and folder. On Colab the next two
cells fetch `piles.py`, `calibrate.py` and the sample data. If the files are
already next to this notebook, for example when you cloned the folder or run
it locally, nothing is downloaded.

In [1]:
import urllib.request
from pathlib import Path

REPO = "fuellabs-ai/hobbylm-embed-cookbook"
BRANCH = "main"
FOLDER = "what-people-ask-for"

FILES = [
    "piles.py",
    "calibrate.py",
    "examples/feature_requests/feature_requests.csv",
    "examples/feature_requests/labelled_sample.csv",
    "examples/feature_requests/piles.yaml",
]


def local_or_fetch(path: str) -> Path:
    local = Path(path)
    if local.exists():
        print(f"using local {path}")
        return local
    url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/{FOLDER}/{path}"
    print(f"downloading {url}")
    local.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, local)
    return local


for name in FILES:
    local_or_fetch(name)

using local piles.py
using local calibrate.py
using local examples/feature_requests/feature_requests.csv
using local examples/feature_requests/labelled_sample.csv
using local examples/feature_requests/piles.yaml


## Dependencies

Colab already has most of this. The cell installs anything missing.

In [2]:
import importlib
import subprocess
import sys

for package in ("sentence_transformers", "sklearn", "pandas"):
    if importlib.util.find_spec(package) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"],
            check=True,
        )

## Load the model

HobbyLM-Embed 140M through Sentence Transformers with `trust_remote_code=True`.
Requests are embedded as plain text with no `prompt_name`, because clustering
needs no query/passage distinction. The vectors are unit length, so a dot
product is cosine similarity.

In [3]:
import warnings

warnings.filterwarnings("ignore")

import transformers

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

from piles import load_model

model = load_model()
print(type(model).__name__)

W1121 03:42:27.768000 83582 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


SentenceTransformer


## Read the export

`piles.py` reads a CSV, picks the text column with `--column` (or a good
guess), and drops empty rows and exact duplicates before anything else.

In [4]:
from piles import clean_requests, embed, pick_column, read_rows

csv_path = local_or_fetch("examples/feature_requests/feature_requests.csv")
rows = read_rows(csv_path)
column = pick_column(rows, "text")
texts, stats = clean_requests(rows, column)
print(f"text column: {column}")
print(stats)
vectors = embed(texts, model)
print(f"{vectors.shape[0]} vectors of dimension {vectors.shape[1]}")

using local examples/feature_requests/feature_requests.csv
text column: text
{'rows_read': 585, 'empty_dropped': 0, 'exact_duplicates_dropped': 2, 'grouped': 583}


583 vectors of dimension 512


## Said again

The near-duplicate pass groups requests that are almost word for word the
same, each cluster shown with a count.

In [5]:
from piles import DEFAULT_CLUSTER_SIM, DEFAULT_DEDUP_SIM, DEFAULT_MIN_GROUP, segment

result = segment(
    vectors,
    dedup_sim=DEFAULT_DEDUP_SIM,
    cluster_sim=DEFAULT_CLUSTER_SIM,
    min_group=DEFAULT_MIN_GROUP,
)
for cluster in result["said_again"][:6]:
    print(f"{cluster['count']} people asked for this: {texts[cluster['example']]}")
print(
    f"... {len(result['said_again'])} said-again clusters covering "
    f"{sum(c['count'] for c in result['said_again'])} of {len(texts)} requests"
)

12 people asked for this: dark mode pls
10 people asked for this: add csv export
9 people asked for this: repeat tasks pls
6 people asked for this: ios app please thanks
5 people asked for this: Google Calendar sync would be perfect, deadlines should live in my calendar
4 people asked for this: Would love add task repetition with an end date, my team runs the same weekly audit
... 130 said-again clusters covering 432 of 583 requests


## Named piles

`piles.yaml` holds piles you name yourself, with 2 to 5 example requests
each. A request joins the pile whose examples it is closest to on average.

In [6]:
from collections import Counter

from piles import DEFAULT_PILE_SIM, parse_piles

piles_path = local_or_fetch("examples/feature_requests/piles.yaml")
piles = parse_piles(piles_path)
result_piles = segment(
    vectors,
    piles=piles,
    model=model,
    pile_sim=DEFAULT_PILE_SIM,
)
for name, count in Counter(result_piles["pile_of"].values()).most_common():
    print(f"pile '{name}': {count}")

using local examples/feature_requests/piles.yaml
pile 'appearance': 74
pile 'getting data out': 60
pile 'on the go': 13


## Groups and keyword labels

Everything left is clustered. Each group gets a label made of its top TF-IDF
keywords. Labels are keywords, not summaries; rename groups in `piles.yaml`
and rerun if you have better names.

In [7]:
from piles import build_groups

groups = build_groups(texts, vectors, result_piles)
for group in groups:
    if group["kind"] == "group":
        top = [texts[i] for i in group["members"][:2]]
        print(f"{len(group['members']):3d} people: {group['name']}")
        for line in top:
            print(f"      {line}")

 15 people: task, tasks, recurring
      Honestly let a recurring task spawn the next one when I check it off would fix most of my days this blocks our rollout
      Add a shortcut to create a task above or below the current one. Every single day this comes up. it is the main gap for us
  9 people: project, tags, tag
      nested tags would help, client then project is all I want
      Adding tag tasks with more than one tag, projects overlap and folders force me to choose would help a lot
  8 people: reminders, notifications, push
      Honestly please notify me before a deadline, not on the day, a day before would be great would fix most of my days we benchmark every tool against this
      Reminders would be useful. I forget deadlines because nothing pings me
  8 people: search, save, decisions
      shortcut to open search with / like every other app would make my week this blocks our rollout
      search in comments too, decisions live in comment threads is all I want this blocks 

## Outputs

`report.html` is the readable report, sorted by size with expandable groups.
`groups.csv` has one row per request with its group and score.

In [8]:
from piles import build_report_html, write_groups_csv

write_groups_csv(Path("groups.csv"), texts, groups)
Path("report.html").write_text(
    build_report_html(csv_path.name, stats, texts, groups, piles is not None),
    encoding="utf-8",
)
print("wrote groups.csv and report.html; open report.html in a browser")

wrote groups.csv and report.html; open report.html in a browser


## Check the settings on the labelled sample

`calibrate.py` sweeps group-size and similarity settings on the 50-row
hand-labelled sample and prints how well the groups match the labels
(adjusted Rand index: 1.0 is a perfect match). The trimmed grid below keeps
the cell quick; run `python calibrate.py examples` locally for the full one.

In [9]:
import calibrate

calibrate.DEDUP_GRID = [0.88, 0.92]
calibrate.CLUSTER_GRID = [0.65, 0.70, 0.75, 0.80]
calibrate.MIN_GROUP_GRID = [3]
sample = Path("examples/feature_requests/labelled_sample.csv")
calibrate.run_sample(sample, model)

=== feature_requests (50 labelled requests, 19 true groups) ===


  min_group 3
  cluster_sim   0.65   0.70   0.75   0.80
  dedup 0.88     0.17   0.52   0.54   0.35
  dedup 0.92     0.25   0.58   0.73   0.59
  suggested: dedup_sim 0.92, cluster_sim 0.75, min_group 3  ARI 0.731, purity 0.776, 26 predicted groups; said-again recall 0.00, purity 1.00



([(0.88, 0.65, 3, 0.17365518283469325, 0.149812734082397, 8),
  (0.88, 0.7, 3, 0.5154451847365233, 0.4935064935064935, 17),
  (0.88, 0.75, 3, 0.5401210047057385, 0.7941176470588235, 29),
  (0.88, 0.8, 3, 0.35150873478030703, 1.0, 39),
  (0.92, 0.65, 3, 0.24753938661023211, 0.18770226537216828, 6),
  (0.92, 0.7, 3, 0.5762099251816243, 0.4811320754716981, 14),
  (0.92, 0.75, 3, 0.73051512592519, 0.7758620689655172, 26),
  (0.92, 0.8, 3, 0.5872641509433962, 1.0, 35)],
 {'best': (0.92, 0.75, 3, 0.73051512592519, 0.7758620689655172, 26),
  'name': 'feature_requests'})

## Limits

- Groups are based on similar wording and meaning, not on what's true or
  important. A count is how many people asked, not how important it is. Small
  groups can be noise, and a request that mixes two asks lands in one group,
  so check each group before it goes on the roadmap. Group labels are
  keywords, not summaries.
- English only.
- "Doesn't fit any group" means unlike the rest of the file, not new or
  emerging.

In [10]:
import platform

import numpy
import sklearn
import torch

print(f"python {platform.python_version()}")
print(f"numpy {numpy.__version__}, torch {torch.__version__}, "
      f"scikit-learn {sklearn.__version__}")

python 3.15.2
numpy 2.5.5, torch 2.15.0, scikit-learn 1.9.1
